In [24]:
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import pacf

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor


from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import time
import glob
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# Read file

In [ ]:
file_pattern = 'raw/observations_part_00[0-3].parquet'
all_files = sorted(glob.glob(file_pattern))

raw_files_hf = [
    "raw/observations_part_000.parquet",
    "raw/observations_part_001.parquet",
    "raw/observations_part_002.parquet",
    "raw/observations_part_003.parquet"
]

if len(all_files) < 4:
    print("Dataset Raw lokal belum ada/belum lengkap. Mengunduh dataset dari Hugging Face...")
    for file in raw_files_hf:
        print(f"-> Mengunduh {file}...")
        hf_hub_download(
            repo_id="vel740/dhaka-smart-waste-processed",
            filename=file,
            repo_type="dataset",
            local_dir="./" 
        )
    
    all_files = sorted(glob.glob(file_pattern))
    print("Dataset selesai diunduh")
else:
    print("Dataset Raw lokal sudah ada/sudah lengkap.")

df_raw = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)

print(f"Total Baris Data: {df_raw.shape[0]:,}")

Dataset Raw lokal belum ada/belum lengkap. Mengunduh dataset dari Hugging Face...
-> Mengunduh raw/observations_part_000.parquet...
-> Mengunduh raw/observations_part_001.parquet...
-> Mengunduh raw/observations_part_002.parquet...
-> Mengunduh raw/observations_part_003.parquet...
Dataset selesai diunduh
Total Baris Data: 1,920,000


# Data Info


In [26]:
df_raw.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1920000 entries, 0 to 1919999
Data columns (total 59 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   activity_multiplier           float32
 1   animal_residue_pct            int32  
 2   battery_level_percent         float32
 3   battery_maintenance_event     int32  
 4   bin_index                     int32  
 5   collection_delay_minutes      float32
 6   collection_event              int32  
 7   collection_threshold_percent  float32
 8   college_calendar_closed       int32  
 9   date_index                    int32  
 10  day_of_week                   int32  
 11  dominant_waste_code           int32  
 12  event_code                    int32  
 13  fill_level_percent            float32
 14  fish_meat_pct                 int32  
 15  friday_effect_active          int32  
 16  green_other_pct               int32  
 17  gross_waste_generation_kg     float32
 18  holiday_code          

In [27]:
df_raw.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,sensor_error_code,temperature_c,textile_pct,time_slot_code,timestamp_unix,true_fill_level_percent,waste_bulk_density_kg_m3,waste_entering_bin_kg,waste_volume_liter,weather_code
0,0.6984,2,99.235924,0,0,0.0,0,80.0,0,0,...,0,15.414943,7,0,1735668000,16.486521,223.214020,0.619539,2.775539,0
1,0.5626,2,95.667847,0,1,0.0,0,80.0,0,0,...,0,13.902746,4,0,1735668000,8.435885,254.822449,0.669809,2.628534,0
2,0.8924,1,99.278976,0,2,0.0,0,80.0,0,0,...,0,13.915423,5,0,1735668000,9.069288,239.105713,1.441714,6.029608,0
3,0.2134,4,97.271172,0,3,0.0,0,75.0,0,0,...,0,14.510163,5,0,1735668000,18.645182,214.397324,0.291495,1.359603,0
4,0.8924,0,95.238556,0,4,0.0,0,75.0,0,0,...,0,13.992124,4,0,1735668000,13.859707,243.774689,3.172023,13.012112,0


# Cleaning Data
Ambil 50 tempat sampah dan urutkan


In [28]:
df_50 = df_raw[df_raw['bin_index'] < 50].sort_values(['bin_index', 'timestamp_unix']).reset_index(drop=True)

## Anomaly Handing
Hapus nilai jika sensor error/packet loss, lalu lakukan interpolasi (pengisian nilai tengah)

In [29]:
anomaly_mask = (df_50['is_sensor_anomaly'] == 1) | (df_50['is_packet_loss'] == 1)
print(f"-> Ditemukan {anomaly_mask.sum()} baris anomali/error hardware. Melakukan Interpolasi...")
df_50.loc[anomaly_mask, 'fill_level_percent'] = np.nan
df_50['fill_level_percent'] = df_50.groupby('bin_index')['fill_level_percent'].transform(
    lambda x: x.interpolate(method='linear').bfill().ffill()
)

-> Ditemukan 131 baris anomali/error hardware. Melakukan Interpolasi...


### Statistical Justification
Ubah nlags dari 5 menjadi 48 (2 hari) atau bahkan 168 (1 minggu)

Dibatasi 48 jam agar output tidak terlalu panjang, cukup melihat tren harian


In [30]:
pacf_vals = pacf(df_50[df_50['bin_index'] == 0]['fill_level_percent'].dropna(), nlags=48)
print(f"Uji PACF (Partial Autocorrelation) pada Bin 0:")
for i, val in enumerate(pacf_vals):
    print(f"  Lag {i} jam lalu: {val:.3f} (Jika > 0.1, artinya signifikan secara statistik)")

Uji PACF (Partial Autocorrelation) pada Bin 0:
  Lag 0 jam lalu: 1.000 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 1 jam lalu: 0.913 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 2 jam lalu: 0.005 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 3 jam lalu: -0.018 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 4 jam lalu: -0.057 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 5 jam lalu: 0.004 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 6 jam lalu: -0.050 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 7 jam lalu: -0.043 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 8 jam lalu: -0.048 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 9 jam lalu: -0.027 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 10 jam lalu: -0.024 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 11 jam lalu: -0.037 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 12 jam lalu: -0.044 (Jika > 0.1, artinya signifik

In [31]:
df_proc = df_50.copy()

#### A. Cyclical Time Encoding (Mencegah AI bingung perpindahan Jam 23 ke Jam 0)

In [32]:
df_proc['hour_sin'] = np.sin(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['hour_cos'] = np.cos(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['day_sin'] = np.sin(2 * np.pi * df_proc['day_of_week'] / 7.0)
df_proc['day_cos'] = np.cos(2 * np.pi * df_proc['day_of_week'] / 7.0)

#### B. Lag Features


In [33]:
df_proc['lag_1h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(1)
df_proc['lag_2h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(2)
df_proc['lag_3h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(3)

df_proc['lag_24h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(24)
df_proc['lag_48h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(48)

df_proc['is_weekend'] = (df_proc['day_of_week'] >= 5).astype(int)

#### C. Ekstraksi Velocity (Laju percepatan sampah masuk)


In [34]:
df_proc['velocity_1h'] = df_proc['fill_level_percent'] - df_proc['lag_1h']
df_proc['velocity_2h'] = df_proc['lag_1h'] - df_proc['lag_2h']

#### D. Pembuatan Target Output (Kapasitas 1 Jam ke depan)


In [35]:
df_proc['target_next_hour'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(-1)

##### Buang NaN


In [36]:
df_proc = df_proc.dropna(subset=['lag_48h','target_next_hour' ]).reset_index(drop=True)

df_proc['target_diff'] = df_proc['target_next_hour'] - df_proc['fill_level_percent']
df_proc['velocity_3h'] = df_proc['lag_2h'] - df_proc['lag_3h']

clean_mask = (
    (df_proc['target_diff'] > -20.0) &
    (df_proc['velocity_1h'] > -20.0) &
    (df_proc['velocity_2h'] > -20.0) &
    (df_proc['velocity_3h'] > -20.0) 
)

df_proc = df_proc[clean_mask].reset_index(drop=True)
print(f'Baris bersih siap training: {len(df_proc):,}')

Baris bersih siap training: 15,650


### Feature Selection

In [37]:
# Set 1: FE Minimal (Hanya 1 Jam memori & Waktu Mentah)
fe_set_1 = [
    'fill_level_percent', 'lag_1h', 'hour', 'day_of_week'
]

# Set 2: FE Kompleks (Multi-Lag, Velocity, & Cyclical Time)
fe_set_2 = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h', 
    'lag_24h', 'lag_48h', 'is_weekend',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]

Mengambil sampel 15000 baris agar komputasi MI cepat


In [38]:
n_sample = min(15000, len(df_proc))
sampel_mi = df_proc.sample(n=n_sample, random_state=42)

mi_scores = mutual_info_regression(sampel_mi[fe_set_2], sampel_mi['target_next_hour'], random_state=42)
mi_df = pd.DataFrame({'Fitur': fe_set_2, 'Skor_MI': mi_scores}).sort_values('Skor_MI', ascending=False)
print("Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):")
print(mi_df.to_string(index=False))

Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):
             Fitur  Skor_MI
fill_level_percent 2.039198
            lag_1h 1.856930
            lag_2h 1.672389
            lag_3h 1.527049
           lag_24h 0.316433
           lag_48h 0.192171
           day_sin 0.033520
        is_weekend 0.011775
           day_cos 0.007219
       velocity_2h 0.003466
       velocity_1h 0.000000
          hour_cos 0.000000
          hour_sin 0.000000


## Train-Test Split


In [39]:
split_timestamp = df_proc['timestamp_unix'].quantile(0.8)

train_mask = df_proc['timestamp_unix'] <= split_timestamp
test_mask = df_proc['timestamp_unix'] > split_timestamp

X_train = df_proc.loc[train_mask, fe_set_2]
y_train = df_proc.loc[train_mask, 'target_next_hour']

X_test = df_proc.loc[test_mask, fe_set_2]
y_test = df_proc.loc[test_mask, 'target_next_hour']

print(f"Batas waktu split: {pd.to_datetime(split_timestamp, unit='s')}")
print(f"Data Training: {len(X_train):,} baris")
print(f"Data Training: {len(X_test):,} baris")

Batas waktu split: 2025-01-13 21:00:00
Data Training: 12,549 baris
Data Training: 3,101 baris


## Experimental A (Using Raw Data)

In [40]:
# Murni data mentah dari sensor tanpa memori
basic_cols = ['fill_level_percent', 'hour', 'day_of_week']

rf_baseline = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_baseline.fit(df_proc.loc[train_mask, basic_cols], y_train)

pred_basic = rf_baseline.predict(df_proc.loc[test_mask, basic_cols])
mae_basic = mean_absolute_error(y_test, pred_basic)

## Experimental B (Using FE Set 1 vs FE Set 2)

In [41]:
# Test FE Set 1
rf_fe1 = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_fe1.fit(df_proc.loc[train_mask, fe_set_1], y_train)
mae_fe1 = mean_absolute_error(y_test, rf_fe1.predict(df_proc.loc[test_mask, fe_set_1]))

# Test FE Set 2
rf_fe2 = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_fe2.fit(df_proc.loc[train_mask, fe_set_2], y_train)
mae_fe2 = mean_absolute_error(y_test, rf_fe2.predict(df_proc.loc[test_mask, fe_set_2]))

print("=== EVALUASI REKAYASA FITUR ===")
print(f"1. Tanpa Feature Engineering (Mentah) : MAE {mae_basic:.3f}%")
print(f"2. Feature Engineering Set 1 (Minimal): MAE {mae_fe1:.3f}%")
print(f"3. Feature Engineering Set 2 (Kompleks): MAE {mae_fe2:.3f}%")

=== EVALUASI REKAYASA FITUR ===
1. Tanpa Feature Engineering (Mentah) : MAE 2.079%
2. Feature Engineering Set 1 (Minimal): MAE 1.937%
3. Feature Engineering Set 2 (Kompleks): MAE 1.664%


## Experimental C (Comprehensive Model & Scaler Comparison)
Menguji Linear Regression, Random Forest, dan XGBoost terhadap berbagai jenis Feature Scaling.


In [42]:
# 1. Dictionary of Scalers
scalers_dict = {
    'Tanpa Scaler': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# 2. Dictionary of Models
models_dict = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
}

grid_results = []
print("Mulai melatih model secara efisien... Mohon tunggu.")

# Ekstrak data mentah untuk model Tree
X_tr_raw = df_proc.loc[train_mask, fe_set_2]
X_te_raw = df_proc.loc[test_mask, fe_set_2]

# Latih semua model Tree HANYA SATU KALI (Tanpa Scaler) untuk menghemat waktu
tree_results = {}
for model_name, model_obj in models_dict.items():
    if model_name != 'Linear Regression':
        start_time = time.time()
        model_obj.fit(X_tr_raw, y_train)
        pred = model_obj.predict(X_te_raw)
        duration = time.time() - start_time
        mae_score = mean_absolute_error(y_test, pred)
        tree_results[model_name] = f"{mae_score:.3f}% ({duration:.2f}s)"

# Bangun baris demi baris untuk tabel
for scaler_name, scaler_obj in scalers_dict.items():
    row_result = {'Scaler': scaler_name}
    
    # Proses Linear Regression (Dijalankan untuk semua scaler)
    if scaler_obj is None:
        X_tr, X_te = X_tr_raw, X_te_raw
    else:
        X_tr = scaler_obj.fit_transform(X_tr_raw)
        X_te = scaler_obj.transform(X_te_raw)
        
    start_time = time.time()
    lr_model = LinearRegression()
    lr_model.fit(X_tr, y_train)
    pred = lr_model.predict(X_te)
    duration = time.time() - start_time
    mae_score = mean_absolute_error(y_test, pred)
    row_result['Linear Regression'] = f"{mae_score:.3f}% ({duration:.2f}s)"
    
    # Isi kolom model Tree
    for model_name in models_dict.keys():
        if model_name != 'Linear Regression':
            if scaler_name == 'Tanpa Scaler':
                row_result[model_name] = tree_results[model_name]
            else:
                row_result[model_name] = "--- Dilewati ---"
                
    grid_results.append(row_result)

Mulai melatih model secara efisien... Mohon tunggu.


# Evaluation Result

In [43]:
# Tampilkan Tabel Grid Hasil Scaler & Model Lanjutan
print("\n=== ARENA KOMPARASI: 6 MODEL vs 4 SCALER (OPTIMIZED) ===")
print("* Format isi tabel: [Nilai Meleset MAE] (Waktu Training & Prediksi)")
print("* Catatan: Pengujian Scaler pada model Tree dilewati untuk menghindari computational noise.\n")

df_grid_results = pd.DataFrame(grid_results)
df_grid_results.set_index('Scaler', inplace=True)
print(df_grid_results.to_string())


=== ARENA KOMPARASI: 6 MODEL vs 4 SCALER (OPTIMIZED) ===
* Format isi tabel: [Nilai Meleset MAE] (Waktu Training & Prediksi)
* Catatan: Pengujian Scaler pada model Tree dilewati untuk menghindari computational noise.

               Linear Regression     Decision Tree     Random Forest           XGBoost          LightGBM          CatBoost
Scaler                                                                                                                    
Tanpa Scaler      1.785% (0.01s)    2.334% (0.24s)    1.662% (2.79s)    1.668% (0.40s)    1.607% (0.20s)    1.615% (5.00s)
StandardScaler    1.785% (0.01s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---
MinMaxScaler      1.785% (0.00s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---
RobustScaler      1.785% (0.00s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---


# Save Dataset

In [44]:
df_proc.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,velocity_1h,velocity_2h,target_next_hour,target_diff,velocity_3h
0,0.6984,2,97.860802,0,0,0.0,0,80.0,1,2,...,68.518791,66.028748,67.102852,42.061382,14.483168,-0.808800,2.490044,70.403191,2.693199,-1.074104
1,0.6984,0,97.832153,0,0,0.0,0,80.0,1,2,...,67.709991,68.518791,66.028748,42.492905,15.378024,2.693199,-0.808800,69.584000,-0.819191,2.490044
2,0.6984,2,97.803505,0,0,0.0,0,80.0,1,2,...,70.403191,67.709991,68.518791,44.316532,16.866007,-0.819191,2.693199,72.692299,3.108299,-0.808800
3,0.6984,0,97.774857,0,0,0.0,0,80.0,1,2,...,69.584000,70.403191,67.709991,43.484684,18.462954,3.108299,-0.819191,68.450325,-4.241974,2.693199
4,0.6984,2,97.746208,0,0,0.0,0,80.0,1,2,...,72.692299,69.584000,70.403191,44.515327,20.921236,-4.241974,3.108299,72.475067,4.024742,-0.819191


In [45]:
trainable_feat = ['bin_index', 'timestamp_unix'] + fe_set_2 + ['target_next_hour']

df_clean = df_proc[trainable_feat].copy()

df_clean.to_csv('./dataset_model/dataset_trainable_rev2.csv', index=False)

print(f"Sudah memasuki ke dalam bentuk csv {df_clean.shape[1]} kolom, siap training")


Sudah memasuki ke dalam bentuk csv 16 kolom, siap training


In [46]:
df_clean.head()

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709991,68.518791,66.028748,67.102852,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403191
1,0,1735844400,70.403191,67.709991,68.518791,66.028748,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.403191,67.709991,68.518791,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692299
3,0,1735851600,72.692299,69.584000,70.403191,67.709991,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.692299,69.584000,70.403191,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475067
